# Ingest Races.csv file

1. Read the file using spark dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"


## Step 1 - Read the CSV file using the dataframe API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

races_schema = StructType([
    StructField('season',    IntegerType()),
    StructField('round',     IntegerType()),
    StructField('url',       StringType()),
    StructField('raceName',  StringType()),
    StructField('date',      DateType()),
    StructField('circuitId', StringType())
])

In [0]:
races_df = (
    spark.read
        .format('csv')
        .option('header', 'true')
        #.option('inferSchema', 'true')
        .option('mode', 'FAILFAST')
        .schema(races_schema)
        .load(source_file)
)

In [0]:
display(races_df)

## Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
races_final_df = add_injestion_metadata(races_df)

In [0]:
display(races_final_df)

## Write to bronze delta table

In [0]:
(
    races_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))